# Topical Lectures, Controls
## Real drone, part 3: our position loop, motors off

Andreas Freise, 01.11.2026

### Goal

So far the drone flew under the controllers of its firmware. From here on the position loop runs in our own Python code on the laptop: the drone reports its position and velocity by radio, our code turns the distance to the target into a velocity, the feedback signal, which sets the velocity setpoint of the firmware's inner loops. The firmware keeps running the velocity loop, the attitude loop and the height; we run the outermost loop, for the position in `x` and `y`. The code is below, the controller of the morning with one PID per axis. It still lacks three numbers, and we will use this notebook and the two that follow to find them.

In this notebook we will find **the sign of the gains**, with an experiment while the motors are off. The drone is moved by hand while the code runs, and the feedback must point back to the start.

Two additional numbers are not in the code but are important to set the correct loop gains: the response time of the drone with its dead time, measured in the flight under the firmware, and the noise of the position estimate, which the first phase of the protocol below shows and the hover of that flight measures again.

The notebook has two milestones. After the first, the direction of the axes and the noise of the estimate are known. After the second, the sign of the gains is known.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = None
URI = crazyflie.drone_uri(LABEL)


## The code that will fly the drone

This is the controller that will fly the drone, defined once here with the gains left open. `make_controller` builds one `PIDController` per axis from the three gains and returns `position_controller`. In flight, `crazyflie.fly` calls that function fifty times a second with the measured position and velocity and the target, and sends the two feedback values it returns to the drone as velocity setpoints. Run the cell; it only defines the code. You enter the gains yourself wherever the code is used, in this notebook and before every flight.


In [ ]:
dt = 0.02        # s, the loop runs at 50 Hz
v_max = 0.3      # m/s, the limit on the feedback, a velocity

def make_controller(Kp, Kd, Ki):
    """One PID per axis with these gains. Returns the controller that fly() calls and the reset of its integrators."""
    x_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)
    y_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)

    def position_controller(x, y, vx, vy, x_target, y_target):
        """Position and velocity in, feedback (a velocity) out, clipped to v_max."""
        vx_fb = x_ctrl(x, vx, x_target)
        vy_fb = y_ctrl(y, vy, y_target)
        return float(np.clip(vx_fb, -v_max, v_max)), float(np.clip(vy_fb, -v_max, v_max))

    def reset_controllers():
        """Zero the integrators, called before every flight."""
        x_ctrl.reset()
        y_ctrl.reset()

    return position_controller, reset_controllers


## Status check

Before anything else we check that the drone is ready: battery, flow deck and supervisor. Switch the drone on, put it on the desk and run the cell. It reads the status over the radio and prints the battery voltage, whether the flow deck is detected and the flags of the supervisor. Run it again whenever something looks wrong.

* `flow deck: NOT attached`: the deck is not seated properly.
* `is locked: yes`: the supervisor has locked the motors after a tumble, a crash or a hard landing, which happens more often on a hard floor. Run the cell after this one; it clears the lock and shows the status again. If it stays locked, switch the drone off and on.
* `No drone answers at ...`: the drone is off, or not on the channel of its label.
* `WARNING: the position estimate runs away` or `WARNING: the estimate does not follow the height sensor`: the flow deck has stopped delivering. Switch the drone off and on, connect again.


In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)

In [ ]:
# Only when the status says locked: clear the lock, then check again.
with crazyflie.connect(URI) as scf:
    crazyflie.recover(scf)
    crazyflie.status(scf)

## The protocol

Now we measure what the drone reports as its position with the motors off: in which direction its axes count, and how noisy the estimate is when nothing moves. You move the drone by hand through five phases while the laptop records the estimate and the readings of the flow deck.

Put the drone on the desk with its forward side, the side between the motors M1 and M4, pointing away from you, so that the two blue LEDs at the back point towards you, and keep it that way during the whole test. No tape measure is needed: a hand cannot move a drone over a known distance to the centimetre, so the protocol does not try to calibrate the estimate.

The cell below talks you through the five phases and counts down the seconds of each. It takes 30 seconds and stops by itself.

1. **Still, 10 s.** The drone lies on the desk, nobody touches it. This is the measurement of the noise and the drift of the estimate.
2. **Lift, 5 s.** Lift the drone 20 to 30 cm and hold it. The flow deck needs to see the table from this height.
3. **Forward, 6 s.** Move it about 30 cm forward, at the same height, and hold it.
4. **Left, 6 s.** Move it about 30 cm to the left and hold it.
5. **Down, 3 s.** Put it down.

The cell first resets the estimator, which sets the origin to where the drone lies. Only the last two seconds of each phase are used, so arrive early and hold still. At the end the cell analyses the samples and prints the results.


In [ ]:
d, ident = crazyflie.identify_sensors(uri=URI)

## Results

The plots show what was recorded in the five phases. Upper left: the position estimate against time. Lower left: what the flow deck itself reported, the optical flow counts, which must be nonzero while the drone moves, and the surface quality, the number of features the sensor finds on the table. Right: the path in the plane, and the height estimate against the range sensor. The printout above the plots ends with two limits on the gains, which follow from the noise: the loop multiplies the position noise by `|Kp|` and the velocity noise by `|Kd|`, the sum is the jitter of the feedback, and keeping that below 0.05 m/s gives the limits. They are upper bounds, not recommendations.

Without a drone, load the recorded reference test instead: `d = crazyflie.load_sensor_id('data/reference/sensor_id_reference.csv')` and `ident = crazyflie.analyse_sensor_id(d)`.


In [ ]:
t = (d['t_ms'] - d['t_ms'][0]) / 1000.0
fig, ax = plt.subplots(2, 2, figsize=(12, 7.5), gridspec_kw={'width_ratios': [2, 1]})

# the estimate
ax[0, 0].plot(t, d['x'], label='x [m]')
ax[0, 0].plot(t, d['y'], label='y [m]')
ax[0, 0].plot(t, d['z'], color='0.6', lw=0.8, label='height [m]')
for name in ('lift', 'forward', 'left', 'down'):
    m = d['phase'] == name
    if m.any():
        for a in (ax[0, 0], ax[1, 0]):
            a.axvline(t[m][0], color='k', lw=0.5, ls=':')
        ax[0, 0].text(t[m][0], 0.98, name, fontsize=8, rotation=90, va='top', transform=ax[0, 0].get_xaxis_transform())
ax[0, 0].axhline(0, color='k', lw=0.5)
ax[0, 0].set_ylabel('position estimate')
ax[0, 0].legend(loc='upper left')
ax[0, 0].grid()

# what the flow deck itself reported
ax[1, 0].plot(t, d['flow_dx'], label='flow dx [counts]')
ax[1, 0].plot(t, d['flow_dy'], label='flow dy [counts]')
ax[1, 0].plot(t, d['squal'], color='0.4', lw=0.8, label='surface quality')
ax[1, 0].axhline(0, color='k', lw=0.5)
ax[1, 0].set_xlabel('t [s]')
ax[1, 0].set_ylabel('flow deck')
ax[1, 0].legend(loc='upper left')
ax[1, 0].grid()

# the path in the plane
ax[0, 1].plot(d['x'], d['y'], '.', ms=2)
ax[0, 1].plot(0, 0, 'k+', ms=12, mew=2, label='start')
ax[0, 1].set_xlabel('x [m]')
ax[0, 1].set_ylabel('y [m]')
ax[0, 1].set_aspect('equal')
ax[0, 1].legend(loc='upper right')
ax[0, 1].grid()

# the height: estimate against the range sensor
ax[1, 1].plot(t, d['zrange_m'], label='range sensor [m]')
ax[1, 1].plot(t, d['z'], label='estimate z [m]')
ax[1, 1].set_xlabel('t [s]')
ax[1, 1].legend(loc='upper left')
ax[1, 1].grid()
plt.tight_layout()

# the gains multiply the noise onto the feedback: |Kp| times the position noise, |Kd| times the velocity noise
jitter = 0.05                                                   # m/s, the jitter of the feedback we accept from the noise alone
Kp_max = jitter / max(ident['std_x_m'], ident['std_y_m'])
Kd_max = jitter / max(ident['std_vx_mps'], ident['std_vy_mps'])
print(f"noise limits for the gains (feedback jitter below {jitter} m/s): |Kp| < {Kp_max:.1f} 1/s, |Kd| < {Kd_max:.1f}")


### Milestone 1: the axes and the noise

The printout ends with the direction of the axes. The milestone is passed when the forward move increased `x`, the left move increased `y`, and the flow sensor saw the table during the moves. If the flow sensor saw nothing, or the height sensor did not see the lift, the deck is not delivering: switch the drone off and on and run the protocol again.

Record the two noise numbers of the still phase in the cell below: the standard deviation of the position and of the velocity, the larger of the two axes each. They are compared with the noise in flight later.


In [ ]:
# Record your results (milestone 1), read off the printout above:
NOISE_X = None     # m, standard deviation of the position estimate while the drone lay still, the larger of x and y
NOISE_V = None     # m/s, standard deviation of the velocity estimate, the larger of vx and vy

## Using the code, motors off

Now our code connects to the real drone for the first time, and we want to find out the sign of the gains. The motors stay off. `crazyflie.handtest` resets the estimator, so that the origin is where the drone lies, and takes that start as the target. For 30 seconds it reads the position and the velocity, calls `position_controller` and prints the feedback twice a second, with an arrow for its sign. Nothing is sent to the drone.

Now we need to record the result in the next cell. `Kd` and `Ki` play no part in this test, so we leave them at 0.0 for now. But for `Kp` we can now set the correct sign. Set a number with either plus or minus (the size does not matter here, so about 0.5 is fine). 

In [ ]:
Kp = None     # 1/s, feedback (a velocity) per metre of position error; the sign is the question of this test
Kd = 0.0      # feedback per m/s of measured velocity; plays no part in this test
Ki = 0.0      # 1/s^2; plays no part in this test


The following code tests your values. While the test runs, lift the drone 20 to 30 cm, move it about 30 cm forward and hold it there for a few seconds, bring it back, then do the same to the left. Keep it level and pointing the same way.

The controller must try to bring the drone back to the start. When you hold the drone 30 cm forward, its position is `x = +0.3` and the target is `x = 0`, so the correct feedback is a velocity backwards, a negative `vx`; the arrow in the printout then shows a minus. The same holds to the left for `vy`. At the end the printout judges each axis by the fraction of the samples beyond 5 cm in which the feedback pointed back towards the start; 80 percent or more is a PASS.

In [ ]:
position_controller, reset_controllers = make_controller(Kp, Kd, Ki)
reset_controllers()
d = crazyflie.handtest(position_controller, uri=URI, duration=30.0)


### Milestone 2: the sign of the feedback

The milestone is passed when both axes print PASS. FAIL on both axes means the sign of `Kp` is wrong: flip it and run the test again. In flight, a loop with the wrong sign would push the drone away from its target, the harder the further away it is. FAIL on one axis only means that the estimate counts that axis the other way round; tell an instructor before you fly. "Not tested" means the drone was not moved far enough on that axis.

Record the sign in the cell below. `KP_SIGN`, `NOISE_X` and `NOISE_V` are the results of this notebook; the next notebooks start with the same three variables, and you copy your values in there.


In [ ]:
# Record your result (milestone 2):
KP_SIGN = None     # +1 or -1, the sign of the gains that the hand test confirmed